# Graph building blocks: examples

Runnable versions of the `PLAN.md` §3.3 examples that the current code supports. Everything runs on CPU with random data, so the numbers show the code works, not that a model is any good.

1. Wrap any model as a one-node graph
2. Train it with your own loop
3. Swap one part; nothing else changes
4. Deploy and predict on one image
5. Mistakes fail early, with the bad name
6. Record a run
7. Evaluate top-1 and top-5

Examples that need code which doesn't exist yet (`Trainer`, data modules, frozen teachers) are still in `PLAN.md` §3.3–3.4. Move each one here when its code lands.

Run from `examples/`. `lwfm` isn't an installed package yet, so put the repository root on the import path.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

In [ ]:
import tempfile

import torch
from torch import nn
from torchvision.models import resnet18
from torchvision.transforms import v2

from lwfm.core.graph import ExecutionGraph, GraphError, ModelNode
from lwfm.core.losses import LossBinding, LossComposer

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)  # ImageNet statistics
_ = torch.manual_seed(0)

## 1. Wrap any model as a one-node graph

The model is an untouched torchvision `resnet18`. `inputs` lists the values passed to `forward`, in order; `"inputs.image"` means `batch["inputs"]["image"]`. `outputs` names what it returns; the node stores it under its own name, as `student.logits`. Preprocessing then the model is a straight chain, so it is plain torch: torchvision's `v2.Resize` and `v2.Normalize` are `nn.Module`s, so they go first in the model's `nn.Sequential`. They resize the 96×96 images to the 64×64 the model is set up for (doing nothing if the size already matches), then normalise. Whatever preprocessing you want goes here; lwfm adds none of its own. The graph is only needed where values branch or merge.

Passing `preflight_sample=batch` runs the graph once when it is built, to check shapes (section 5).

In [ ]:
batch = {
    "inputs": {"image": torch.rand(8, 3, 96, 96)},  # floats in [0, 1]
    "targets": {"class": torch.randint(0, 10, (8,))},
    "meta": {"sample_id": [f"s{i}" for i in range(8)]},
}

student = ModelNode(
    name="student",
    module=nn.Sequential(  # a straight chain: plain torch
        v2.Resize((64, 64), antialias=True),
        v2.Normalize(MEAN, STD),
        resnet18(num_classes=10),  # untouched torchvision model
    ),
    inputs="inputs.image",  # passed to forward as its first argument
    outputs="logits",  # its Tensor return, stored as "student.logits"
)
# preflight_sample: run once on this batch now, to check shapes
pipeline = ExecutionGraph([student], preflight_sample=batch)

out = pipeline(batch)
{name: tuple(value.shape) for name, value in out.items()}

## 2. Train it with your own loop

The pipeline (an `ExecutionGraph`) and the loss composer are enough to train; a `Trainer` will be a convenience, not a requirement. The loss reads `student.logits` and `targets.class`, passed to `CrossEntropyLoss(input, target)` in that order. Overfitting one batch is a quick check that gradients reach the weights: the loss should fall towards 0.

In [ ]:
CE_READS = ["student.logits", "targets.class"]
loss_fn = LossComposer([LossBinding("ce", nn.CrossEntropyLoss(), CE_READS)])
optimizer = torch.optim.SGD(pipeline.parameters(), lr=0.1)

pipeline.train()
for step in range(20):  # overfit one batch: loss should fall
    total, terms = loss_fn(batch, pipeline(batch))
    optimizer.zero_grad()
    total.backward()
    optimizer.step()
    if step in (0, 19):
        print(step, {name: round(t.item(), 4) for name, t in terms.items()})

## 3. Swap one part; nothing else changes

**A different model.** `TinyViT` names its argument `pixels` and returns a dict. It is the last step of the chain, so its dict is what the node receives. Inputs are passed by position, so `inputs` stays the same; `outputs` lists the dict fields to keep. The loss still reads `student.logits`, so it needs no changes. The extra `cls_token` output becomes `student.cls_token`, ready for a later feature-distillation loss.

In [ ]:
class TinyViT(nn.Module):
    def __init__(self, num_classes: int, dim: int = 32, patch: int = 8):
        super().__init__()
        self.embed = nn.Conv2d(3, dim, kernel_size=patch, stride=patch)
        self.cls = nn.Parameter(torch.zeros(1, 1, dim))
        layer = nn.TransformerEncoderLayer(
            dim, nhead=4, dim_feedforward=2 * dim, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(
            layer, num_layers=2, enable_nested_tensor=False
        )
        self.head = nn.Linear(dim, num_classes)

    def forward(self, pixels: torch.Tensor) -> dict[str, torch.Tensor]:
        tokens = self.embed(pixels).flatten(2).transpose(1, 2)  # [B, N, dim]
        tokens = torch.cat([self.cls.expand(len(tokens), -1, -1), tokens], dim=1)
        cls_token = self.encoder(tokens)[:, 0]
        return {"logits": self.head(cls_token), "cls_token": cls_token}


vit_student = ModelNode(
    name="student",
    module=nn.Sequential(
        v2.Resize((64, 64), antialias=True),
        v2.Normalize(MEAN, STD),
        TinyViT(num_classes=10),
    ),
    inputs="inputs.image",
    outputs=["logits", "cls_token"],  # dict fields to keep
)
vit_pipeline = ExecutionGraph([vit_student], preflight_sample=batch)
{name: tuple(value.shape) for name, value in vit_pipeline(batch).items()}

**A different loss, or a weighted mix.** $\mathcal{L} = 0.9\,\mathrm{CE}_{\text{smoothed}} + 0.1\,\mathrm{margin}$. Both are plain `torch.nn` losses, reading the same values. The composer returns the total for `backward()` and each term, detached, for logging.

In [ ]:
mixed_loss = LossComposer(
    [
        LossBinding(
            "ce", nn.CrossEntropyLoss(label_smoothing=0.1), CE_READS, weight=0.9
        ),
        LossBinding("margin", nn.MultiMarginLoss(), CE_READS, weight=0.1),
    ]
)
total, terms = mixed_loss(batch, vit_pipeline(batch))
total, terms

## 4. Deploy and predict on one image

`deployable_model` returns the node's module, here the `nn.Sequential` of preprocessing and model, as a plain `nn.Module`. It contains no graph, names or losses, and it carries its preprocessing, so it takes a raw `[0, 1]` image of any size. Its state dict holds only model weights (`Resize` and `Normalize` store none), so a fresh copy reloads it and gives identical predictions. A random tensor stands in for an image file. For a real image: `read_image(path, mode=ImageReadMode.RGB).float() / 255` from `torchvision.io`.

In [ ]:
model = pipeline.deployable_model("student").eval()  # Resize, Normalize, resnet18

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "model_weights.pt"
    torch.save(model.state_dict(), path)
    fresh = nn.Sequential(
        v2.Resize((64, 64), antialias=True),
        v2.Normalize(MEAN, STD),
        resnet18(num_classes=10),
    )
    fresh.load_state_dict(torch.load(path, weights_only=True))
    fresh.eval()

image = torch.rand(3, 120, 80)  # [3, H, W] in [0, 1], any size
with torch.inference_mode():
    probs, classes = model(image[None]).softmax(-1).topk(5)
    torch.testing.assert_close(fresh(image[None]), model(image[None]))
classes, probs

## 5. Mistakes fail early, with the bad name

The graph checks its wiring when it is built, before any data is touched: every value a node reads must come from the batch (`inputs.`, `targets.` or `meta.`) or from another node, node names must be unique, and there must be no cycles.

In [ ]:
try:
    ExecutionGraph(
        [
            ModelNode("student", resnet18(), inputs="inputs.image", outputs="logits"),
            ModelNode(
                # "studnet" is a deliberate typo: building fails and names it
                "head",
                nn.Linear(512, 10),
                inputs="studnet.features",
                outputs="logits",
            ),
        ]
    )
except GraphError as error:
    print(f"GraphError: {error}")

Wrong **shapes** only show up when data flows. Pass `ExecutionGraph(nodes, preflight_sample=batch)` and building the graph also runs it once on that batch (the *preflight* check), in eval mode without gradients; any batch works, even zeros of the right shape. It also checks every output keeps the batch dimension first. The output shapes are kept in `pipeline.preflight_output_shapes`, and `pipeline.preflight(batch)` re-runs the check. Without a sample the check is skipped. On failure the error keeps PyTorch's type, with a note naming the node and the shapes it read; the same note is added in ordinary `pipeline(batch)` calls, and losses and metrics add one naming themselves.

In [ ]:
import traceback

# checked when the pipeline was built in section 1
print(pipeline.preflight_output_shapes)

probe = ModelNode("probe", nn.Linear(512, 2), inputs="student.logits", outputs="logits")
try:
    ExecutionGraph([student, probe], preflight_sample=batch)  # 512 should be 10
except RuntimeError as error:
    print("".join(traceback.format_exception_only(error)))

## 6. Record a run

Training code **emits** events; **sinks** decide where they go. `LocalFileSink` writes the run directory: `manifest.json` (with versions and git state) and `status.json` straight away, each event to `events.jsonl` as it happens, and `summary.json` (the last value of each scalar) at the end. `ConsoleSink` prints one line per event. `OutputRouter` is itself a sink that sends every event to both and, used with `with`, records the final status: `completed`, `failed` (with the traceback) or `interrupted`. Scalars can be passed as 0-dim tensors, as the loss composer returns them.

In [ ]:
import json

from lwfm.results.events import RunEvent
from lwfm.results.router import OutputRouter
from lwfm.results.sinks import ConsoleSink, LocalFileSink

run_dir = Path(tempfile.mkdtemp()) / "demo_run"

with OutputRouter(
    LocalFileSink(run_dir, manifest_fields={"name": "notebook_demo"}), ConsoleSink()
) as router:
    pipeline.train()
    for step in range(3):
        total, terms = loss_fn(batch, pipeline(batch))
        optimizer.zero_grad()
        total.backward()
        optimizer.step()
        for name, term in terms.items():
            router.write(
                RunEvent("scalar", f"train/loss/{name}", term, stage="fit", step=step)
            )

In [ ]:
print(sorted(p.name for p in run_dir.iterdir()))
print("status:", json.loads((run_dir / "status.json").read_text())["status"])
print("events:", len((run_dir / "events.jsonl").read_text().splitlines()))
print("summary:", json.loads((run_dir / "summary.json").read_text()))

## 7. Evaluate top-1 and top-5

A metric is an unchanged torchmetrics object; a `MetricBinding` names the values its `update(preds, target)` reads, in order. `Evaluator().evaluate` resets every metric, runs the graph in eval mode under `torch.inference_mode()`, updates each metric per batch and computes once at the end, so accuracy is over all samples (micro), not an average of batch averages. The graph is put back in train mode afterwards, so the same call works for validation during training. Any plain list or `DataLoader` of batches works as the loader. With an `output_sink`, values are written as `eval/<split>/<name>`. Both scores are 1.0 here only because the model overfit this very batch in section 2.

In [ ]:
from torchmetrics.classification import MulticlassAccuracy

from lwfm.evaluation.evaluator import Evaluator
from lwfm.evaluation.metrics import MetricBinding

PRED_READS = ["student.logits", "targets.class"]  # update(preds, target)
metrics = [
    MetricBinding(
        f"top{k}",
        MulticlassAccuracy(num_classes=10, top_k=k, average="micro"),
        PRED_READS,
    )
    for k in (1, 5)
]
val_loader = [batch, batch]  # any iterable of batches; usually a DataLoader

result = Evaluator().evaluate(
    pipeline,
    val_loader,
    metrics,
    device=torch.device("cpu"),
    output_sink=ConsoleSink(),  # any OutputSink
)
print(result.metric_values, "| pipeline back in train mode:", pipeline.training)